# uv 与 Python 环境管理

> uv / pyproject.toml / venv / pip / 依赖

## 为什么需要它

Python 生态在 2020 年前后面临一个尴尬局面：

- **依赖打架**：`pip install` 默认装到全局，项目 A 要 `Django==4.2`，项目 B 要 `Django==5.0`，谁先装谁赢，后装的炸。虽然 `venv`（Python 自带的虚拟环境模块）能隔离项目环境，但操作繁琐——创建、激活、装包、冻结，每一步都要人记命令
- **Python 版本本身也在打架**：macOS 自带 Python 3.9，你需要 3.12，还得额外装 **pyenv**（一个管理多 Python 版本的工具）来回切
- **依赖树不可复现**：`requirements.txt` 只记录你直接装的包，而每个包又有自己的依赖（叫**间接依赖**或传递依赖）。今天装的 `requests==2.31.0` 依赖 `urllib3==2.0.6`，三个月后 urllib3 更新到 2.1.0 可能和你的代码不兼容——但 `requirements.txt` 里根本看不出来

**uv**（2024 年由 Astral 团队发布）用 Rust 重写了整个 Python 包管理流程，一个工具统一解决版本管理、虚拟环境、依赖锁定三件事，而且比 pip 快 10-100 倍。

> **Rust**：一种系统编程语言，特点是内存安全且速度接近 C。uv 的底层用 Rust 写，所以极快。

## 是什么

| 概念 | 一句话 |
|------|--------|
| **Python 解释器** | 执行 Python 代码的程序（`python.exe` / `python3`），不同项目可能需不同版本 |
| **venv（虚拟环境）** | 一个独立的 Python 运行环境目录（`.venv/`），里面有自己的一套包，和全局隔离 |
| **pip** | Python 官方包安装器，从 **PyPI**（Python Package Index，Python 公共包仓库网站 pypi.org）下载安装包 |
| **uv** | 新一代 Python 包管理器（Rust 实现），替代 pip + venv + pyenv |
| **pyproject.toml** | 现代 Python 项目的标准配置文件，声明项目元信息、依赖、工具配置 |
| **uv.lock（依赖锁定文件）** | uv 自动生成的精确版本清单，锁死每个直接和间接依赖的版本，保证任何人装出来完全一样 |

### 虚拟环境原理

```text
全局 pip install：                  使用虚拟环境：
  Python312/Lib/site-packages/        项目A/.venv/
    requests 2.31  ← B 要 2.31         requests 2.28  ← A 要 2.28
    Django 5.0     ← A 要 4.2！         Django 4.2
                                      项目B/.venv/
                                        requests 2.31
                                        Django 5.0
```
激活虚拟环境就是把 `.venv/` 里的 `python` 放到 Shell 的 PATH 最前面——之后所有的 `python` 和 `pip` 都指向项目自己的那个。

## 怎么样

### uv 用法

In [ ]:
# === 安装 uv（一次性） ===
# Windows:
powershell -c "irm https://astral.sh/uv/install.ps1 | iex"
# macOS / Linux:
curl -LsSf https://astral.sh/uv/install.sh | sh

# === 安装 Python ===
uv python install 3.12        # 安装指定版本 Python
uv python list                 # 查看已安装的版本

# === 创建项目 ===
uv init my-project             # 生成 pyproject.toml
cd my-project

# === 添加依赖 ===
uv add requests                # 安装并写入 pyproject.toml，生成 uv.lock
uv add --dev pytest            # 开发依赖（测试用，不随项目发布）

# === 运行 ===
uv run python main.py          # 自动用项目虚拟环境执行
uv run pytest


### pip 对比

In [ ]:
# 传统 pip 流程
python -m venv .venv           # 创建虚拟环境

# 激活（Windows）
.venv\Scripts\activate
# 激活（macOS/Linux）
source .venv/bin/activate

pip install requests
pip install -r requirements.txt
pip freeze > requirements.txt  # 冻结版本
deactivate                      # 退出虚拟环境


### pyproject.toml 示例

In [ ]:
[project]
name = "my-app"
version = "0.1.0"
description = "一个示例项目"
requires-python = ">=3.12"
dependencies = [
    "fastapi>=0.115.0",
    "uvicorn[standard]>=0.30.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=8.0",
    "ruff>=0.5",
]

[tool.ruff]
line-length = 100
target-version = "py312"


> `>=0.115.0` 是版本约束：至少 0.115.0 版本，不锁死小版本。`uv.lock` 会把每个包钉死在精确版本（如 `0.115.12`）。

### uv vs pip vs conda

| | pip + venv | uv | conda |
|------|-----------|-----|------|
| 速度 | 慢（纯 Python） | 极快（Rust） | 中 |
| Python 版本管理 | ❌ 需 pyenv | ✅ | ✅ |
| 依赖锁定 | ❌（requirements.txt 不锁间接依赖） | ✅ uv.lock | ✅ environment.yml |
| 生态 | 纯 Python | 纯 Python | Python + C 库等非 Python 依赖 |
| 适合 | 旧项目兼容 | 现代 Python 项目（推荐） | 数据科学/ML |

## 相关笔记

- [01-管理工具/07-Shell与终端基础.ipynb](01-管理工具/07-Shell与终端基础.ipynb)——PATH 与虚拟环境激活的关系
- [03-编码与序列化/YAML与TOML.ipynb](../03-编码与序列化/YAML与TOML.ipynb)——TOML 格式详解